# 01. Introduction Section - Homework

In [24]:
# Downloading the data from https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page
# !wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2022-01.parquet

In [25]:
# !wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2022-02.parquet

In [43]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

## Q1. Number of columns in the dataset
### 19

In [27]:
jan_df = pd.read_parquet(r"yellow_tripdata_2022-01.parquet")

In [28]:
n_cols = jan_df.shape[1]
n_cols

19

## Q2. Std of the duration of the trip
### 46.45

In [29]:
jan_df.dtypes

VendorID                          int64
tpep_pickup_datetime     datetime64[ns]
tpep_dropoff_datetime    datetime64[ns]
passenger_count                 float64
trip_distance                   float64
RatecodeID                      float64
store_and_fwd_flag               object
PULocationID                      int64
DOLocationID                      int64
payment_type                      int64
fare_amount                     float64
extra                           float64
mta_tax                         float64
tip_amount                      float64
tolls_amount                    float64
improvement_surcharge           float64
total_amount                    float64
congestion_surcharge            float64
airport_fee                     float64
dtype: object

In [32]:
jan_df["duration"] = jan_df.tpep_dropoff_datetime - jan_df.tpep_pickup_datetime
jan_df["duration"] = jan_df["duration"].dt.total_seconds() / 60

std_trip = np.std(jan_df["duration"])
round(std_trip, 2)

46.45

## Q3. Counting the number of samples after drop outliers
### 98.275

In [34]:
jan_df["duration"]

0          17.816667
1           8.400000
2           8.966667
3          10.033333
4          37.533333
             ...    
2463926     5.966667
2463927    10.650000
2463928    11.000000
2463929    12.050000
2463930    27.000000
Name: duration, Length: 2463931, dtype: float64

In [36]:
n_samples_before_drop_outliers = jan_df.shape[0]
jan_df = jan_df.loc[(jan_df["duration"] > 1) & (jan_df["duration"] <= 60)]
n_samples_after_drop_outliers = jan_df.shape[0]


percent_samples_after_drop_outliers = n_samples_after_drop_outliers / n_samples_before_drop_outliers
round(percent_samples_after_drop_outliers, 5) * 100

98.275

## Q4. Dimensionality of the data after One-hot encoding
### 2

In [38]:
jan_df.columns

Index(['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime',
       'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag',
       'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra',
       'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge',
       'total_amount', 'congestion_surcharge', 'airport_fee', 'duration'],
      dtype='object')

In [39]:
categorical_cols = ["PULocationID", "DOLocationID"]

train_dict = jan_df[categorical_cols].to_dict(orient="records")

dv = DictVectorizer()

X_train = dv.fit_transform(train_dict)

n_cols_in_X_train = X_train.shape[1]
n_cols_in_X_train

2

## Q5. Getting the RMSE on train after Training a model
### 8.920327827581444

In [42]:
target = "duration"
y_train = jan_df[target].values

In [44]:
linear_reg_model = LinearRegression()
linear_reg_model.fit(X_train, y_train)

y_pred_on_train = linear_reg_model.predict(X_train)

rmse_on_train = mean_squared_error(y_train, y_pred_on_train, squared=False)
rmse_on_train

8.920327827581444

## Q6. Evaluating the model on the validation data (Feb)
### 9.638272212087234

In [45]:
feb_df = pd.read_parquet(r"yellow_tripdata_2022-02.parquet")
feb_df["duration"] = feb_df.tpep_dropoff_datetime - feb_df.tpep_pickup_datetime
feb_df["duration"] = feb_df["duration"].dt.total_seconds() / 60
feb_df = feb_df.loc[(feb_df["duration"] > 1) & (feb_df["duration"] <= 60)]

categorical_cols = ["PULocationID", "DOLocationID"]
val_dict = feb_df[categorical_cols].to_dict(orient="records")
X_val = dv.transform(val_dict)

target = "duration"
y_val = feb_df[target].values

y_pred_on_val = linear_reg_model.predict(X_val)

rmse_on_val = mean_squared_error(y_val, y_pred_on_val, squared=False)
rmse_on_val


9.638272212087234